
# Experiment 21 — Strong iTransformer Baseline Reproduction

## 목적

이번 실험에서는 아직 retrieval을 붙이지 않습니다.

먼저 두 번째 strong backbone인 **iTransformer**를 네 데이터셋과 네 prediction horizon에서 재현합니다.

$$
\mathcal{D}
=
\{
\mathrm{ETTm1},
\mathrm{ETTh1},
\mathrm{Weather},
\mathrm{Electricity}
\}
$$

$$
H
\in
\{
96,192,336,720
\}
$$

총 조건 수는

$$
4\times4=16
$$

입니다.

이 baseline이 충분히 강하게 재현된 이후에 다음 Experiment에서

$$
\boxed{
F_{\mathrm{iTransformer}}
+
R_{\mathrm{historical}}
>
F_{\mathrm{iTransformer}}
}
$$

를 검증합니다.

---

## 왜 baseline을 별도로 먼저 확인하는가?

PatchTST 실험에서 ETTm1의 초기 baseline이 공식 recipe보다 상당히 약하다는 것을 발견했습니다.

따라서 iTransformer에서도 retrieval augmentation을 붙이기 전에 direct backbone 자체를 먼저 고정합니다.

이렇게 하면 이후 retrieval improvement가

- weak baseline 때문인지,
- 실제 historical memory의 complementary information 때문인지

구분할 수 있습니다.

---

## iTransformer protocol

### Canonical lookback

iTransformer의 공식 long-term forecasting 실험은 기본적으로

$$
L_D=96
$$

을 사용합니다.

이는 PatchTST에서 사용한 \(L_D=336\)과 다르지만,
이번 연구의 핵심 비교는 backbone 간 절대 성능 비교가 아니라

$$
F+R
\quad\text{vs.}\quad
F
$$

의 **within-backbone improvement**입니다.

따라서 각 backbone의 공식적이고 강한 recipe를 유지합니다.

---

## Dataset-specific frozen recipes

### ETTm1 / ETTh1

공식 iTransformer `run.py`의 기본 recipe를 사용합니다.

- sequence length: 96
- model dimension: 512
- attention heads: 8
- encoder layers: 2
- FFN dimension: 2048
- dropout: 0.1
- batch size: 32
- learning rate: \(10^{-4}\)
- train epochs: 10
- patience: 3
- LR schedule: type1
- seed: 2023
- normalization inside iTransformer: on

ETTm1은 15-minute time feature를 위해 `freq='t'`를 사용합니다.

### Weather

공식 multivariate forecasting script를 따릅니다.

- sequence length: 96
- encoder layers: 3
- model dimension: 512
- FFN dimension: 512
- heads: 8
- batch size: 32
- learning rate: \(10^{-4}\)
- train epochs: 10
- patience: 3
- LR schedule: type1

### Electricity

공식 ECL multivariate forecasting script를 따릅니다.

- sequence length: 96
- encoder layers: 3
- model dimension: 512
- FFN dimension: 512
- heads: 8
- batch size: 16
- learning rate: \(5\times10^{-4}\)
- train epochs: 10
- patience: 3
- LR schedule: type1

---

## Evaluation

iTransformer 공식 data factory는 test에서 batch size 1을 사용합니다.

따라서 test set의 모든 valid origin을 stride 1로 평가합니다.

이번 notebook에서는 training epoch마다 test loss를 보지 않고,
validation loss만 checkpoint selection에 사용합니다.

이는 공식 optimization을 유지하면서 test inspection을 피하기 위한 더 엄격한 protocol입니다.

---

## 이후 Experiment 22

baseline이 정상적으로 재현되면
기존 frozen historical-memory module을 그대로 적용합니다.

Retrieval 쪽 설정은 변경하지 않습니다.

$$
\text{Predictive Retrieval}
\rightarrow
\text{Uniform Top-10 Historical Forecast}
\rightarrow
\text{Cross-Fitted Adaptive Trust}
\rightarrow
\text{Validation-Calibrated Shrinkage}
$$


In [1]:

from pathlib import Path
from types import SimpleNamespace
import gc
import importlib
import os
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 460)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATASETS = [
    "ETTm1",
    "ETTh1",
    "Weather",
    "Electricity",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

TASKS = [
    (dataset, horizon)
    for dataset in DATASETS
    for horizon in HORIZONS
]

SEQ_LEN = 96
LABEL_LEN = 48
SEED = 2023

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "itransformer_official_baseline_reproduction"
)

CHECKPOINT_DIR = ROOT / "checkpoints"
HISTORY_DIR = ROOT / "history"
ARTIFACT_DIR = ROOT / "artifacts"

for p in [
    ROOT,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    ARTIFACT_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

RESUME = True
FORCE_RETRAIN = False

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Tasks:", len(TASKS))
print("Output:", ROOT)


Device: cuda
PyTorch: 2.12.1+cu126
Tasks: 16
Output: /data/dataset/strong_forecaster/itransformer_official_baseline_reproduction


## 1. Locate or clone the official iTransformer repository

In [2]:

REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/iTransformer_official"
    ),
    Path("/code/iTransformer"),
    Path("/data/iTransformer"),
]

REPO = next(
    (
        p
        for p in REPO_CANDIDATES
        if (
            p / "model" / "iTransformer.py"
        ).is_file()
        and (
            p / "data_provider" / "data_factory.py"
        ).is_file()
    ),
    None,
)

if REPO is None:
    target = REPO_CANDIDATES[0]

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Official iTransformer repository not found."
    )
    print(
        "Attempting to clone into:",
        target,
    )

    try:
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/thuml/iTransformer.git",
                str(target),
            ],
            check=True,
        )
    except Exception as e:
        raise RuntimeError(
            "Could not clone the official iTransformer repository. "
            "Clone https://github.com/thuml/iTransformer.git manually "
            f"into {target} and rerun this cell."
        ) from e

    REPO = target

if not (
    REPO / "model" / "iTransformer.py"
).is_file():
    raise FileNotFoundError(
        f"Official iTransformer model not found under {REPO}"
    )

try:
    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    commit = "unknown"

print("Official repo:", REPO)
print("Commit:", commit)

(ARTIFACT_DIR / "official_repo_commit.txt").write_text(
    commit + "\n"
)


Official iTransformer repository not found.
Attempting to clone into: /code/stock_regime_retrieval/strong_forecaster/iTransformer_official


Cloning into '/code/stock_regime_retrieval/strong_forecaster/iTransformer_official'...


Official repo: /code/stock_regime_retrieval/strong_forecaster/iTransformer_official
Commit: c2426e68ca13f74aaec08045c5c724d8ad328124


41

## 2. Import official model, data provider, and LR utility

In [3]:

# Remove potentially conflicting top-level modules.
for module_name in list(sys.modules.keys()):
    if (
        module_name == "model"
        or module_name.startswith("model.")
        or module_name == "layers"
        or module_name.startswith("layers.")
        or module_name == "data_provider"
        or module_name.startswith("data_provider.")
        or module_name == "utils"
        or module_name.startswith("utils.")
    ):
        del sys.modules[module_name]

if str(REPO) in sys.path:
    sys.path.remove(
        str(REPO)
    )

sys.path.insert(
    0,
    str(REPO),
)

itransformer_module = importlib.import_module(
    "model.iTransformer"
)

data_factory_module = importlib.import_module(
    "data_provider.data_factory"
)

tools_module = importlib.import_module(
    "utils.tools"
)

OfficialITransformer = itransformer_module.Model
official_data_provider = data_factory_module.data_provider
official_adjust_lr = tools_module.adjust_learning_rate

actual_model_file = Path(
    itransformer_module.__file__
).resolve()

expected_model_file = (
    REPO / "model" / "iTransformer.py"
).resolve()

print("Imported model:", actual_model_file)

if actual_model_file != expected_model_file:
    raise RuntimeError(
        "Wrong iTransformer implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

print(
    "PASS: official iTransformer implementation is active."
)


Imported model: /code/stock_regime_retrieval/strong_forecaster/iTransformer_official/model/iTransformer.py
PASS: official iTransformer implementation is active.


## 3. Dataset paths

In [4]:

DATA_PATH_CANDIDATES = {
    "ETTm1": [
        Path("/data/dataset/ETTm1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTm1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTm1.csv"
        ),
    ],
    "ETTh1": [
        Path("/data/dataset/ETTh1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTh1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTh1.csv"
        ),
    ],
    "Weather": [
        Path("/data/dataset/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/weather/weather.csv"
        ),
    ],
    "Electricity": [
        Path("/data/dataset/electricity.csv"),
        Path(
            "/data/dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/electricity/electricity.csv"
        ),
    ],
}

DATA_PATHS = {}

for name, candidates in DATA_PATH_CANDIDATES.items():
    hit = next(
        (
            p
            for p in candidates
            if p.is_file()
        ),
        None,
    )

    DATA_PATHS[name] = hit

    print(
        f"{name:11s}:",
        hit if hit is not None else "NOT FOUND",
    )

missing = [
    name
    for name, path in DATA_PATHS.items()
    if path is None
]

if missing:
    print("\nAttempted paths:")

    for name in missing:
        print(f"\n{name}")

        for p in DATA_PATH_CANDIDATES[name]:
            print(" -", p)

    raise FileNotFoundError(
        "Dataset file(s) not found: "
        + ", ".join(missing)
    )


ETTm1      : /data/dataset/ETTm1.csv
ETTh1      : /data/Time-Series-Library/dataset/ETT-small/ETTh1.csv
Weather    : /data/Time-Series-Library/dataset/weather/weather.csv
Electricity: /data/Time-Series-Library/dataset/electricity/electricity.csv


## 4. Frozen official-style recipes

In [5]:

RECIPES = {
    "ETTm1": {
        "data": "ETTm1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "t",
    },
    "ETTh1": {
        "data": "ETTh1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Weather": {
        "data": "custom",
        "enc_in": 21,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Electricity": {
        "data": "custom",
        "enc_in": 321,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 16,
        "learning_rate": 5e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
}

display(
    pd.DataFrame(RECIPES).T
)


,data,enc_in,e_layers,d_model,d_ff,n_heads,dropout,batch_size,learning_rate,train_epochs,patience,lradj,factor,freq
ETTm1,ETTm1,7,2,512,2048,8,0.1,32,0.0001,10,3,type1,1,t
ETTh1,ETTh1,7,2,512,2048,8,0.1,32,0.0001,10,3,type1,1,h
Weather,custom,21,3,512,512,8,0.1,32,0.0001,10,3,type1,1,h
Electricity,custom,321,3,512,512,8,0.1,16,0.0005,10,3,type1,1,h



## 5. Published iTransformer reference MSE

아래 값은 baseline fidelity를 판단하기 위한 **reference**입니다.

실험 환경과 repository version에 따라 소폭 차이가 날 수 있으므로
이 값 자체를 optimization target으로 사용하지 않습니다.

또한 test result를 보고 hyperparameter를 바꾸지 않습니다.

| Dataset | 96 | 192 | 336 | 720 |
|---|---:|---:|---:|---:|
| ETTm1 | 0.334 | 0.377 | 0.426 | 0.491 |
| ETTh1 | 0.386 | 0.441 | 0.487 | 0.503 |
| Weather | 0.174 | 0.221 | 0.278 | 0.358 |
| Electricity | 0.148 | 0.162 | 0.178 | 0.225 |

이번 Experiment의 목적은 이 값에 맞추어 tuning하는 것이 아니라
**고정된 official-style recipe가 충분히 강하게 재현되는지 확인하는 것**입니다.


In [6]:

REFERENCE_MSE = {
    ("ETTm1", 96): 0.334,
    ("ETTm1", 192): 0.377,
    ("ETTm1", 336): 0.426,
    ("ETTm1", 720): 0.491,

    ("ETTh1", 96): 0.386,
    ("ETTh1", 192): 0.441,
    ("ETTh1", 336): 0.487,
    ("ETTh1", 720): 0.503,

    ("Weather", 96): 0.174,
    ("Weather", 192): 0.221,
    ("Weather", 336): 0.278,
    ("Weather", 720): 0.358,

    ("Electricity", 96): 0.148,
    ("Electricity", 192): 0.162,
    ("Electricity", 336): 0.178,
    ("Electricity", 720): 0.225,
}


## 6. Build official argument namespace

In [7]:

def make_args(
    name,
    horizon,
):
    r = RECIPES[name]
    path = DATA_PATHS[name]

    return SimpleNamespace(
        # basic
        is_training=1,
        model_id=f"{name}_96_{horizon}",
        model="iTransformer",

        # data
        data=r["data"],
        root_path=str(path.parent) + "/",
        data_path=path.name,
        features="M",
        target="OT",
        freq=r["freq"],
        checkpoints=str(CHECKPOINT_DIR),

        # forecasting
        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=int(horizon),

        # model
        enc_in=r["enc_in"],
        dec_in=r["enc_in"],
        c_out=r["enc_in"],
        d_model=r["d_model"],
        n_heads=r["n_heads"],
        e_layers=r["e_layers"],
        d_layers=1,
        d_ff=r["d_ff"],
        moving_avg=25,
        factor=r["factor"],
        distil=True,
        dropout=r["dropout"],
        embed="timeF",
        activation="gelu",
        output_attention=False,
        do_predict=False,

        # optimization
        num_workers=10,
        itr=1,
        train_epochs=r["train_epochs"],
        batch_size=r["batch_size"],
        patience=r["patience"],
        learning_rate=r["learning_rate"],
        des="Exp",
        loss="MSE",
        lradj=r["lradj"],
        use_amp=False,

        # GPU
        use_gpu=torch.cuda.is_available(),
        gpu=0,
        use_multi_gpu=False,
        devices="0",

        # iTransformer
        exp_name="MTSF",
        channel_independence=False,
        inverse=False,
        class_strategy="projection",
        target_root_path="./data/electricity/",
        target_data_path="electricity.csv",
        efficient_training=False,
        use_norm=1,
        partial_start_index=0,
    )


## 7. Data protocol preflight

In [8]:

protocol_rows = []

for name, horizon in TASKS:
    args = make_args(
        name,
        horizon,
    )

    train_data, train_loader = official_data_provider(
        args,
        "train",
    )

    val_data, val_loader = official_data_provider(
        args,
        "val",
    )

    test_data, test_loader = official_data_provider(
        args,
        "test",
    )

    protocol_rows.append({
        "Dataset": name,
        "Horizon": horizon,
        "TrainWindows": len(train_data),
        "ValWindows": len(val_data),
        "TestWindows": len(test_data),
        "TrainBatches": len(train_loader),
        "ValBatches": len(val_loader),
        "TestBatches": len(test_loader),
        "TrainBatchSize": args.batch_size,
        "TestBatchSize": 1,
    })

    del (
        train_data,
        train_loader,
        val_data,
        val_loader,
        test_data,
        test_loader,
    )

protocol_df = pd.DataFrame(protocol_rows)

display(protocol_df)

protocol_df.to_csv(
    ARTIFACT_DIR / "data_protocol.csv",
    index=False,
)


train 34369
val 11425
test 11425
train 34273
val 11329
test 11329
train 34129
val 11185
test 11185
train 33745
val 10801
test 10801
train 8449
val 2785
test 2785
train 8353
val 2689
test 2689
train 8209
val 2545
test 2545
train 7825
val 2161
test 2161
train 36696
val 5175
test 10444
train 36600
val 5079
test 10348
train 36456
val 4935
test 10204
train 36072
val 4551
test 9820
train 18221
val 2537
test 5165
train 18125
val 2441
test 5069
train 17981
val 2297
test 4925
train 17597
val 1913
test 4541


,Dataset,Horizon,TrainWindows,ValWindows,TestWindows,TrainBatches,ValBatches,TestBatches,TrainBatchSize,TestBatchSize
0,ETTm1,96,34369,11425,11425,1074,357,11425,32,1
1,ETTm1,192,34273,11329,11329,1071,354,11329,32,1
2,ETTm1,336,34129,11185,11185,1066,349,11185,32,1
3,ETTm1,720,33745,10801,10801,1054,337,10801,32,1
4,ETTh1,96,8449,2785,2785,264,87,2785,32,1
5,ETTh1,192,8353,2689,2689,261,84,2689,32,1
6,ETTh1,336,8209,2545,2545,256,79,2545,32,1
7,ETTh1,720,7825,2161,2161,244,67,2161,32,1
8,Weather,96,36696,5175,10444,1146,161,10444,32,1
9,Weather,192,36600,5079,10348,1143,158,10348,32,1


## 8. Model construction and parameter count

In [9]:

def build_model(
    name,
    horizon,
):
    args = make_args(
        name,
        horizon,
    )

    model = OfficialITransformer(
        args
    ).float().to(
        DEVICE
    )

    return (
        model,
        args,
    )


param_rows = []

for name in DATASETS:
    model, args = build_model(
        name,
        96,
    )

    params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    param_rows.append({
        "Dataset": name,
        "Params": params,
        "Params_M": params / 1e6,
        "Layers": args.e_layers,
        "d_model": args.d_model,
        "d_ff": args.d_ff,
        "Heads": args.n_heads,
    })

    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

display(
    pd.DataFrame(param_rows)
)


,Dataset,Params,Params_M,Layers,d_model,d_ff,Heads
0,ETTm1,6404704,6.404704,2,512,2048,8
1,ETTh1,6404704,6.404704,2,512,2048,8
2,Weather,4833888,4.833888,3,512,512,8
3,Electricity,4833888,4.833888,3,512,512,8


## 9. Training/evaluation utilities

In [10]:

def set_seed(
    seed=SEED,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False


def checkpoint_path(
    name,
    horizon,
):
    return (
        CHECKPOINT_DIR
        / f"{name}_H{horizon}_seed{SEED}.pt"
    )


def history_path(
    name,
    horizon,
):
    return (
        HISTORY_DIR
        / f"{name}_H{horizon}_seed{SEED}.csv"
    )


@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    args,
):
    model.eval()

    batch_mse = []

    sse = 0.0
    sae = 0.0
    count = 0
    windows = 0

    for (
        batch_x,
        batch_y,
        batch_x_mark,
        batch_y_mark,
    ) in loader:
        batch_x = batch_x.float().to(
            DEVICE
        )

        batch_y = batch_y.float().to(
            DEVICE
        )

        if (
            "PEMS" in args.data
            or "Solar" in args.data
        ):
            batch_x_mark = None
            batch_y_mark = None
        else:
            batch_x_mark = batch_x_mark.float().to(
                DEVICE
            )

            batch_y_mark = batch_y_mark.float().to(
                DEVICE
            )

        dec_inp = torch.zeros_like(
            batch_y[
                :,
                -args.pred_len:,
                :
            ]
        ).float()

        dec_inp = torch.cat(
            [
                batch_y[
                    :,
                    :args.label_len,
                    :
                ],
                dec_inp,
            ],
            dim=1,
        ).float().to(
            DEVICE
        )

        outputs = model(
            batch_x,
            batch_x_mark,
            dec_inp,
            batch_y_mark,
        )

        outputs = outputs[
            :,
            -args.pred_len:,
            :
        ]

        true = batch_y[
            :,
            -args.pred_len:,
            :
        ]

        err = outputs - true

        batch_mse.append(
            float(
                (err ** 2).mean().item()
            )
        )

        sse += float(
            (err ** 2).sum().item()
        )

        sae += float(
            err.abs().sum().item()
        )

        count += err.numel()
        windows += len(batch_x)

        del (
            batch_x,
            batch_y,
            batch_x_mark,
            batch_y_mark,
            dec_inp,
            outputs,
            true,
            err,
        )

    return {
        "BatchAverageMSE": float(
            np.mean(batch_mse)
        ),
        "MSE": sse / count,
        "MAE": sae / count,
        "Windows": windows,
        "Batches": len(batch_mse),
        "Elements": count,
    }



## 10. Training loop

공식 iTransformer optimization을 그대로 따릅니다.

$$
\mathcal{L}
=
\mathrm{MSE}
$$

Optimizer는 Adam이고,
`type1` learning-rate schedule은 매 epoch 이후 learning rate를 절반으로 줄입니다.

Checkpoint 선택에는 validation loss만 사용합니다.

Test set은 best checkpoint가 확정된 이후 한 번만 평가합니다.


In [11]:

def train_or_load(
    name,
    horizon,
):
    path = checkpoint_path(
        name,
        horizon,
    )

    model, args = build_model(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE_RETRAIN
    ):
        ckpt = torch.load(
            path,
            map_location=DEVICE,
        )

        model.load_state_dict(
            ckpt["StateDict"]
        )

        model.eval()

        print(
            f"Loaded {name} H={horizon}: "
            f"best={ckpt['BestValMSE']:.6f}"
            f"@{ckpt['BestEpoch']}"
        )

        return (
            model,
            args,
            ckpt,
        )

    set_seed(
        SEED
    )

    train_data, train_loader = official_data_provider(
        args,
        "train",
    )

    val_data, val_loader = official_data_provider(
        args,
        "val",
    )

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=args.learning_rate,
    )

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0

    history = []

    for epoch in range(
        1,
        args.train_epochs + 1,
    ):
        model.train()

        losses = []
        t0 = time.time()

        for (
            batch_x,
            batch_y,
            batch_x_mark,
            batch_y_mark,
        ) in train_loader:
            optimizer.zero_grad(
                set_to_none=True
            )

            batch_x = batch_x.float().to(
                DEVICE
            )

            batch_y = batch_y.float().to(
                DEVICE
            )

            if (
                "PEMS" in args.data
                or "Solar" in args.data
            ):
                batch_x_mark = None
                batch_y_mark = None
            else:
                batch_x_mark = batch_x_mark.float().to(
                    DEVICE
                )

                batch_y_mark = batch_y_mark.float().to(
                    DEVICE
                )

            dec_inp = torch.zeros_like(
                batch_y[
                    :,
                    -args.pred_len:,
                    :
                ]
            ).float()

            dec_inp = torch.cat(
                [
                    batch_y[
                        :,
                        :args.label_len,
                        :
                    ],
                    dec_inp,
                ],
                dim=1,
            ).float().to(
                DEVICE
            )

            outputs = model(
                batch_x,
                batch_x_mark,
                dec_inp,
                batch_y_mark,
            )

            outputs = outputs[
                :,
                -args.pred_len:,
                :
            ]

            true = batch_y[
                :,
                -args.pred_len:,
                :
            ]

            loss = F.mse_loss(
                outputs,
                true,
            )

            loss.backward()
            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            del (
                batch_x,
                batch_y,
                batch_x_mark,
                batch_y_mark,
                dec_inp,
                outputs,
                true,
                loss,
            )

        train_mse = float(
            np.mean(losses)
        )

        val = evaluate_loader(
            model,
            val_loader,
            args,
        )

        val_mse = val[
            "BatchAverageMSE"
        ]

        if (
            best_state is None
            or val_mse <= best_val
        ):
            best_val = val_mse
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }

            wait = 0
        else:
            wait += 1

        # Match official type1 scheduler.
        official_adjust_lr(
            optimizer,
            epoch,
            args,
        )

        current_lr = optimizer.param_groups[
            0
        ][
            "lr"
        ]

        history.append({
            "Epoch": epoch,
            "TrainMSE": train_mse,
            "ValBatchAverageMSE": val_mse,
            "ValGlobalMSE": val["MSE"],
            "ValMAE": val["MAE"],
            "BestValMSE": best_val,
            "BestEpoch": best_epoch,
            "LR": current_lr,
            "Seconds": time.time() - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            history_path(
                name,
                horizon,
            ),
            index=False,
        )

        print(
            f"{name:11s} H={horizon:3d} "
            f"ep={epoch:02d}/{args.train_epochs} | "
            f"train={train_mse:.6f} | "
            f"val={val_mse:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"lr={current_lr:.3e} | "
            f"wait={wait}/{args.patience}"
        )

        if wait >= args.patience:
            print(
                f"Early stopping: {name} H={horizon}"
            )
            break

    if best_state is None:
        raise RuntimeError(
            f"{name} H={horizon}: no best checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Dataset": name,
        "Horizon": int(horizon),
        "SeqLen": SEQ_LEN,
        "Seed": SEED,
        "BestEpoch": int(best_epoch),
        "BestValMSE": float(best_val),
        "Recipe": RECIPES[name],
        "OfficialRepoCommit": commit,
        "StateDict": best_state,
    }

    torch.save(
        ckpt,
        path,
    )

    del (
        train_data,
        train_loader,
        val_data,
        val_loader,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        model,
        args,
        ckpt,
    )



## 11. Run all 16 baseline conditions

계산이 싼 ETT부터 시작하고,
Weather와 Electricity를 뒤에 실행합니다.

각 조건이 완료되는 즉시 checkpoint와 summary를 저장합니다.

중간에 kernel이 종료되어도

```python
RESUME = True
```

상태에서 다시 실행하면 완료된 조건은 자동으로 skip됩니다.


In [12]:

SUMMARY_PATH = ROOT / "summary.csv"

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

result_rows = (
    existing.to_dict("records")
    if len(existing)
    else []
)


def already_done(
    name,
    horizon,
):
    if not len(existing):
        return False

    return bool(
        (
            (
                existing["Dataset"]
                == name
            )
            & (
                existing["Horizon"]
                == horizon
            )
        ).any()
    )


for name, horizon in TASKS:
    if already_done(
        name,
        horizon,
    ):
        print(
            f"SKIP completed: "
            f"{name} H={horizon}"
        )
        continue

    print(
        "\n"
        + "=" * 140
    )

    print(
        f"OFFICIAL iTransformer | "
        f"{name} | L=96 | H={horizon}"
    )

    print(
        "=" * 140
    )

    t0 = time.time()

    model, args, ckpt = train_or_load(
        name,
        horizon,
    )

    test_data, test_loader = official_data_provider(
        args,
        "test",
    )

    test = evaluate_loader(
        model,
        test_loader,
        args,
    )

    ref_mse = REFERENCE_MSE[
        (
            name,
            horizon,
        )
    ]

    row = {
        "Dataset": name,
        "Horizon": horizon,
        "SeqLen": SEQ_LEN,
        "Channels": RECIPES[name]["enc_in"],
        "BestEpoch": ckpt["BestEpoch"],
        "BestValMSE": ckpt["BestValMSE"],
        "Test_MSE": test["MSE"],
        "Test_MAE": test["MAE"],
        "TestWindows": test["Windows"],
        "Reference_MSE": ref_mse,
        "AbsDiffVsReference": (
            test["MSE"]
            - ref_mse
        ),
        "RelDiffVsReference_pct": (
            100.0
            * (
                test["MSE"]
                - ref_mse
            )
            / ref_mse
        ),
        "RuntimeMinutes": (
            time.time()
            - t0
        )
        / 60.0,
        "Checkpoint": str(
            checkpoint_path(
                name,
                horizon,
            )
        ),
        "OfficialRepoCommit": commit,
    }

    result_rows.append(
        row
    )

    pd.DataFrame(
        result_rows
    ).to_csv(
        SUMMARY_PATH,
        index=False,
    )

    display(
        pd.DataFrame(
            [row]
        )[
            [
                "Dataset",
                "Horizon",
                "BestEpoch",
                "Test_MSE",
                "Test_MAE",
                "Reference_MSE",
                "RelDiffVsReference_pct",
                "RuntimeMinutes",
            ]
        ]
    )

    del (
        model,
        args,
        ckpt,
        test_data,
        test_loader,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(
        result_rows
    )
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    summary_df
)



OFFICIAL iTransformer | ETTm1 | L=96 | H=96
train 34369
val 11425
Updating learning rate to 0.0001
ETTm1       H= 96 ep=01/10 | train=0.296938 | val=0.396834 | best=0.396834@1 | lr=1.000e-04 | wait=0/3
Updating learning rate to 5e-05
ETTm1       H= 96 ep=02/10 | train=0.254466 | val=0.399854 | best=0.396834@1 | lr=5.000e-05 | wait=1/3
Updating learning rate to 2.5e-05
ETTm1       H= 96 ep=03/10 | train=0.229127 | val=0.397187 | best=0.396834@1 | lr=2.500e-05 | wait=2/3
Updating learning rate to 1.25e-05
ETTm1       H= 96 ep=04/10 | train=0.214624 | val=0.410467 | best=0.396834@1 | lr=1.250e-05 | wait=3/3
Early stopping: ETTm1 H=96
test 11425


,Dataset,Horizon,BestEpoch,Test_MSE,Test_MAE,Reference_MSE,RelDiffVsReference_pct,RuntimeMinutes
0,ETTm1,96,1,0.336712,0.373422,0.334,0.811988,1.265152



OFFICIAL iTransformer | ETTm1 | L=96 | H=192
train 34273
val 11329
Updating learning rate to 0.0001
ETTm1       H=192 ep=01/10 | train=0.344616 | val=0.520437 | best=0.520437@1 | lr=1.000e-04 | wait=0/3
Updating learning rate to 5e-05
ETTm1       H=192 ep=02/10 | train=0.309371 | val=0.522029 | best=0.520437@1 | lr=5.000e-05 | wait=1/3
Updating learning rate to 2.5e-05
ETTm1       H=192 ep=03/10 | train=0.285923 | val=0.516765 | best=0.516765@3 | lr=2.500e-05 | wait=0/3
Updating learning rate to 1.25e-05
ETTm1       H=192 ep=04/10 | train=0.272389 | val=0.518865 | best=0.516765@3 | lr=1.250e-05 | wait=1/3
Updating learning rate to 6.25e-06
ETTm1       H=192 ep=05/10 | train=0.265341 | val=0.521158 | best=0.516765@3 | lr=6.250e-06 | wait=2/3
Updating learning rate to 3.125e-06
ETTm1       H=192 ep=06/10 | train=0.261573 | val=0.522317 | best=0.516765@3 | lr=3.125e-06 | wait=3/3
Early stopping: ETTm1 H=192
test 11329


,Dataset,Horizon,BestEpoch,Test_MSE,Test_MAE,Reference_MSE,RelDiffVsReference_pct,RuntimeMinutes
0,ETTm1,192,3,0.39438,0.402333,0.377,4.610062,1.762467



OFFICIAL iTransformer | ETTm1 | L=96 | H=336
train 34129
val 11185
Updating learning rate to 0.0001
ETTm1       H=336 ep=01/10 | train=0.397292 | val=0.659234 | best=0.659234@1 | lr=1.000e-04 | wait=0/3
Updating learning rate to 5e-05
ETTm1       H=336 ep=02/10 | train=0.361608 | val=0.651977 | best=0.651977@2 | lr=5.000e-05 | wait=0/3
Updating learning rate to 2.5e-05
ETTm1       H=336 ep=03/10 | train=0.336342 | val=0.659974 | best=0.651977@2 | lr=2.500e-05 | wait=1/3
Updating learning rate to 1.25e-05
ETTm1       H=336 ep=04/10 | train=0.321941 | val=0.665626 | best=0.651977@2 | lr=1.250e-05 | wait=2/3
Updating learning rate to 6.25e-06
ETTm1       H=336 ep=05/10 | train=0.314321 | val=0.667694 | best=0.651977@2 | lr=6.250e-06 | wait=3/3
Early stopping: ETTm1 H=336
test 11185


...


OFFICIAL iTransformer | Electricity | L=96 | H=336
train 17981
val 2297
Updating learning rate to 0.0005
Electricity H=336 ep=01/10 | train=0.226724 | val=0.174150 | best=0.174150@1 | lr=5.000e-04 | wait=0/3
Updating learning rate to 0.00025
Electricity H=336 ep=02/10 | train=0.200291 | val=0.165166 | best=0.165166@2 | lr=2.500e-04 | wait=0/3
Updating learning rate to 0.000125
Electricity H=336 ep=03/10 | train=0.190001 | val=0.160060 | best=0.160060@3 | lr=1.250e-04 | wait=0/3
Updating learning rate to 6.25e-05
Electricity H=336 ep=04/10 | train=0.183997 | val=0.159867 | best=0.159867@4 | lr=6.250e-05 | wait=0/3
Updating learning rate to 3.125e-05
Electricity H=336 ep=05/10 | train=0.179301 | val=0.157588 | best=0.157588@5 | lr=3.125e-05 | wait=0/3
Updating learning rate to 1.5625e-05
Electricity H=336 ep=06/10 | train=0.176300 | val=0.157253 | best=0.157253@6 | lr=1.563e-05 | wait=0/3
Updating learning rate to 7.8125e-06
Electricity H=336 ep=07/10 | train=0.174605 | val=0.157445 | b

,Dataset,Horizon,BestEpoch,Test_MSE,Test_MAE,Reference_MSE,RelDiffVsReference_pct,RuntimeMinutes
0,Electricity,336,9,0.179687,0.272384,0.178,0.948014,5.916747



OFFICIAL iTransformer | Electricity | L=96 | H=720
train 17597
val 1913
Updating learning rate to 0.0005
Electricity H=720 ep=01/10 | train=0.278834 | val=0.208237 | best=0.208237@1 | lr=5.000e-04 | wait=0/3
Updating learning rate to 0.00025
Electricity H=720 ep=02/10 | train=0.246764 | val=0.207882 | best=0.207882@2 | lr=2.500e-04 | wait=0/3
Updating learning rate to 0.000125
Electricity H=720 ep=03/10 | train=0.230724 | val=0.204815 | best=0.204815@3 | lr=1.250e-04 | wait=0/3
Updating learning rate to 6.25e-05
Electricity H=720 ep=04/10 | train=0.219446 | val=0.209536 | best=0.204815@3 | lr=6.250e-05 | wait=1/3
Updating learning rate to 3.125e-05
Electricity H=720 ep=05/10 | train=0.213007 | val=0.210860 | best=0.204815@3 | lr=3.125e-05 | wait=2/3
Updating learning rate to 1.5625e-05
Electricity H=720 ep=06/10 | train=0.209759 | val=0.212724 | best=0.204815@3 | lr=1.563e-05 | wait=3/3
Early stopping: Electricity H=720
test 4541


,Dataset,Horizon,BestEpoch,Test_MSE,Test_MAE,Reference_MSE,RelDiffVsReference_pct,RuntimeMinutes
0,Electricity,720,3,0.217054,0.304202,0.225,-3.531769,4.182344


,Dataset,Horizon,SeqLen,Channels,BestEpoch,BestValMSE,Test_MSE,Test_MAE,TestWindows,Reference_MSE,AbsDiffVsReference,RelDiffVsReference_pct,RuntimeMinutes,Checkpoint,OfficialRepoCommit
0,ETTh1,96,96,7,1,0.684389,0.392304,0.407334,2785,0.386,0.006304,1.633050,0.367928,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
1,ETTh1,192,96,7,1,0.991093,0.442639,0.434737,2689,0.441,0.001639,0.371734,0.379621,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
2,ETTh1,336,96,7,1,1.274273,0.489263,0.461457,2545,0.487,0.002263,0.464646,0.370271,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
3,ETTh1,720,96,7,1,1.550578,0.506507,0.492894,2161,0.503,0.003507,0.697193,0.349391,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
4,ETTm1,96,96,7,1,0.396834,0.336712,0.373422,11425,0.334,0.002712,0.811988,1.265152,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
5,ETTm1,192,96,7,3,0.516765,0.394380,0.402333,11329,0.377,0.017380,4.610062,1.762467,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
6,ETTm1,336,96,7,2,0.651977,0.424327,0.422129,11185,0.426,-0.001673,-0.392829,1.558324,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
7,ETTm1,720,96,7,3,0.966520,0.496095,0.461646,10801,0.491,0.005095,1.037615,1.796674,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
8,Electricity,96,96,321,7,0.128056,0.148275,0.239707,5165,0.148,0.000275,0.185944,5.385768,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124
9,Electricity,192,96,321,7,0.139029,0.165100,0.256106,5069,0.162,0.003100,1.913884,5.600795,/data/dataset/strong_forecaster/itransformer_o...,c2426e68ca13f74aaec08045c5c724d8ad328124


## 12. Compact fidelity table

In [13]:

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "Test_MSE",
        "Test_MAE",
        "Reference_MSE",
        "RelDiffVsReference_pct",
        "BestEpoch",
        "RuntimeMinutes",
    ]
].copy()

compact["FidelityBand"] = pd.cut(
    compact["RelDiffVsReference_pct"].abs(),
    bins=[
        -np.inf,
        3.0,
        7.5,
        np.inf,
    ],
    labels=[
        "Close (<=3%)",
        "Moderate (3-7.5%)",
        "Large (>7.5%)",
    ],
)

display(
    compact
)

compact.to_csv(
    ROOT / "compact_fidelity_table.csv",
    index=False,
)


,Dataset,Horizon,Test_MSE,Test_MAE,Reference_MSE,RelDiffVsReference_pct,BestEpoch,RuntimeMinutes,FidelityBand
0,ETTh1,96,0.392304,0.407334,0.386,1.633050,1,0.367928,Close (<=3%)
1,ETTh1,192,0.442639,0.434737,0.441,0.371734,1,0.379621,Close (<=3%)
2,ETTh1,336,0.489263,0.461457,0.487,0.464646,1,0.370271,Close (<=3%)
3,ETTh1,720,0.506507,0.492894,0.503,0.697193,1,0.349391,Close (<=3%)
4,ETTm1,96,0.336712,0.373422,0.334,0.811988,1,1.265152,Close (<=3%)
5,ETTm1,192,0.394380,0.402333,0.377,4.610062,3,1.762467,Moderate (3-7.5%)
6,ETTm1,336,0.424327,0.422129,0.426,-0.392829,2,1.558324,Close (<=3%)
7,ETTm1,720,0.496095,0.461646,0.491,1.037615,3,1.796674,Close (<=3%)
8,Electricity,96,0.148275,0.239707,0.148,0.185944,7,5.385768,Close (<=3%)
9,Electricity,192,0.165100,0.256106,0.162,1.913884,7,5.600795,Close (<=3%)



## 13. Interpretation rule

### 충분히 가까운 재현

reference 대비 오차가 몇 % 수준이고,
전체적인 horizon ordering과 magnitude가 일치하면 baseline을 고정합니다.

### 한두 조건이 다소 차이

seed 또는 repository version 차이일 수 있습니다.

이 경우 test result에 맞추어 hyperparameter를 바꾸지 않고
validation history와 protocol을 먼저 점검합니다.

### 여러 조건에서 7.5% 이상 크게 약함

retrieval experiment로 바로 넘어가지 않습니다.

먼저 다음을 확인합니다.

- dataset split
- time-feature frequency
- normalization
- official repository import
- learning-rate schedule
- checkpoint selection
- official recipe parameter

---

## 중요한 원칙

Experiment 21의 test result를 보고 hyperparameter search를 하지 않습니다.

필요한 경우 공개 official script와 protocol parity를 점검한 뒤
명백한 implementation mismatch만 수정합니다.

baseline이 확정되면 Experiment 22에서 historical-memory augmentation을 적용합니다.


## 14. Runtime summary

In [14]:

if len(summary_df):
    runtime_summary = (
        summary_df
        .groupby(
            "Dataset",
            as_index=False,
        )
        .agg(
            Conditions=(
                "Horizon",
                "size",
            ),
            TotalRuntimeMinutes=(
                "RuntimeMinutes",
                "sum",
            ),
            MeanRuntimeMinutes=(
                "RuntimeMinutes",
                "mean",
            ),
        )
    )

    runtime_summary[
        "TotalRuntimeHours"
    ] = (
        runtime_summary[
            "TotalRuntimeMinutes"
        ]
        / 60.0
    )

    display(
        runtime_summary
    )

    print(
        "Completed total runtime (hours):",
        summary_df[
            "RuntimeMinutes"
        ].sum()
        / 60.0,
    )


,Dataset,Conditions,TotalRuntimeMinutes,MeanRuntimeMinutes,TotalRuntimeHours
0,ETTh1,4,1.467212,0.366803,0.024454
1,ETTm1,4,6.382617,1.595654,0.106377
2,Electricity,4,21.085654,5.271414,0.351428
3,Weather,4,10.357365,2.589341,0.172623


Completed total runtime (hours): 0.6548807981279161


## 15. Saved artifacts

In [15]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob("*")
):
    if p.is_file():
        print(
            p.relative_to(
                ROOT
            )
        )


Experiment root: /data/dataset/strong_forecaster/itransformer_official_baseline_reproduction
artifacts/data_protocol.csv
artifacts/official_repo_commit.txt
checkpoints/ETTh1_H192_seed2023.pt
checkpoints/ETTh1_H336_seed2023.pt
checkpoints/ETTh1_H720_seed2023.pt
checkpoints/ETTh1_H96_seed2023.pt
checkpoints/ETTm1_H192_seed2023.pt
checkpoints/ETTm1_H336_seed2023.pt
checkpoints/ETTm1_H720_seed2023.pt
checkpoints/ETTm1_H96_seed2023.pt
checkpoints/Electricity_H192_seed2023.pt
checkpoints/Electricity_H336_seed2023.pt
checkpoints/Electricity_H720_seed2023.pt
checkpoints/Electricity_H96_seed2023.pt
checkpoints/Weather_H192_seed2023.pt
checkpoints/Weather_H336_seed2023.pt
checkpoints/Weather_H720_seed2023.pt
checkpoints/Weather_H96_seed2023.pt
compact_fidelity_table.csv
history/ETTh1_H192_seed2023.csv
history/ETTh1_H336_seed2023.csv
history/ETTh1_H720_seed2023.csv
history/ETTh1_H96_seed2023.csv
history/ETTm1_H192_seed2023.csv
history/ETTm1_H336_seed2023.csv
history/ETTm1_H720_seed2023.csv
histor